In [ ]:
import pandas as pd
import plotly.express as px

# ── Load your results dataframe ──────────────────────────────────────────────
# Replace this with however you load results_df in your notebook, e.g.:
# results_df = pd.read_csv('bridge_results.csv')
# Or if it's already in memory, just make sure results_df is defined above this script.

# Required columns:
#   LAT_016, LONG_017, prob_fail_5yr, prob_fail_10yr, prob_fail_20yr,
#   STRUCTURE_NUMBER_008, COUNTY_CODE_003, PLACE_CODE_004, risk_category

# ── Choose time horizon ───────────────────────────────────────────────────────
# Change this to 'prob_fail_5yr' or 'prob_fail_20yr' as needed
HORIZON = 'prob_fail_10yr'
HORIZON_LABEL = '10-Year'  # Update to match HORIZON

results_df = pd.read_csv("RESULTS_prob_of_failure.csv")

print(results_df[['LAT_016','LONG_017',HORIZON]].dtypes)
print(results_df.shape)


# ── Filter to valid coordinates ───────────────────────────────────────────────
plot_df = results_df.dropna(subset=['LAT_016', 'LONG_017', HORIZON]).copy()
print(plot_df.shape)
print(plot_df[[ 'LAT_016','LONG_017']].head())


In [ ]:
import pandas as pd
import plotly.express as px

# ── Load your results dataframe ──────────────────────────────────────────────
# Replace this with however you load results_df in your notebook, e.g.:
# results_df = pd.read_csv('bridge_results.csv')
# Or if it's already in memory, just make sure results_df is defined above this script.

# Required columns:
#   LAT_016, LONG_017, prob_fail_5yr, prob_fail_10yr, prob_fail_20yr,
#   STRUCTURE_NUMBER_008, COUNTY_CODE_003, PLACE_CODE_004, risk_category

# ── Choose time horizon ───────────────────────────────────────────────────────
# Change this to 'prob_fail_5yr' or 'prob_fail_20yr' as needed
HORIZON = 'prob_fail_10yr'
HORIZON_LABEL = '10-Year'  # Update to match HORIZON

# ── Convert NBI packed DMS coordinates to decimal degrees ────────────────────
# NBI stores coordinates as packed integers e.g. 41593540 = 41° 59' 35.40"
def nbi_to_decimal(val):
    val = int(val)
    degrees  = val // 1000000
    minutes  = (val % 1000000) // 10000
    seconds  = (val % 10000) / 100
    return degrees + minutes / 60 + seconds / 3600

plot_df = results_df.dropna(subset=['LAT_016', 'LONG_017', HORIZON]).copy()
plot_df['lat_dd'] = plot_df['LAT_016'].apply(nbi_to_decimal)
plot_df['lon_dd'] = -plot_df['LONG_017'].apply(nbi_to_decimal)  # West = negative

# ── Fix data types ────────────────────────────────────────────────────────────
plot_df['YEAR_BUILT_027']      = plot_df['YEAR_BUILT_027'].astype('Int64')
plot_df['COUNTY_CODE_003']     = plot_df['COUNTY_CODE_003'].astype('Int64')
plot_df['PLACE_CODE_004']      = plot_df['PLACE_CODE_004'].astype('Int64')
plot_df['predicted_year_goes_poor'] = plot_df['predicted_year_goes_poor'].astype('Int64')
plot_df['years_remaining']     = plot_df['years_remaining'].astype('Int64')

# ── Massachusetts bounding box sanity filter ──────────────────────────────────
plot_df = plot_df[
    (plot_df['lat_dd'].between(41.4, 42.95)) &
    (plot_df['lon_dd'].between(-73.6, -69.9))
]

# ── Hover text ────────────────────────────────────────────────────────────────
plot_df['hover'] = (
    'Structure: ' + plot_df['STRUCTURE_NUMBER_008'].astype(str) +
    '<br>County: ' + plot_df['COUNTY_CODE_003'].astype(str) +
    '<br>Place: ' + plot_df['PLACE_CODE_004'].astype(str) +
    '<br>Year Built: ' + plot_df['YEAR_BUILT_027'].astype(str) +
    '<br>Predicted Year Poor: ' + plot_df['predicted_year_goes_poor'].astype(str) +
    '<br>Years Remaining: ' + plot_df['years_remaining'].astype(str) +
    '<br>5yr risk: ' + (plot_df['prob_fail_5yr'] * 100).round(1).astype(str) + '%' +
    '<br>10yr risk: ' + (plot_df['prob_fail_10yr'] * 100).round(1).astype(str) + '%' +
    '<br>20yr risk: ' + (plot_df['prob_fail_20yr'] * 100).round(1).astype(str) + '%' +
    '<br>Category: ' + plot_df['risk_category'].astype(str)
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = px.scatter_mapbox(
    plot_df,
    lat='lat_dd',
    lon='lon_dd',
    color=HORIZON,
    hover_name='STRUCTURE_NUMBER_008',
    hover_data={'LAT_016': False, 'LONG_017': False, HORIZON: False, 'hover': True},
    color_continuous_scale=[
        [0.0,  '#1D9E75'],   # low risk  – teal
        [0.33, '#EF9F27'],   # medium    – amber
        [0.66, '#E24B4A'],   # high      – red
        [1.0,  '#501313'],   # very high – dark red
    ],
    range_color=[0, 1],
    zoom=7,
    center={'lat': 42.2, 'lon': -71.8},
    mapbox_style='carto-positron',
    title=f'Massachusetts Bridge Failure Risk — {HORIZON_LABEL} Probability',
    labels={HORIZON: f'{HORIZON_LABEL} Failure Probability'},
    height=700,
)

fig.update_traces(
    marker=dict(size=6, opacity=0.8),
    customdata=plot_df[['hover']],
    hovertemplate='%{customdata[0]}<extra></extra>',
)

fig.update_layout(
    coloraxis_colorbar=dict(
        title=f'{HORIZON_LABEL}<br>Failure Prob.',
        tickformat='.0%',
        tickvals=[0, 0.25, 0.5, 0.75, 1.0],
        ticktext=['0%', '25%', '50%', '75%', '100%'],
        len=0.6,
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    font=dict(family='Arial'),
)

fig.show()

# ── Optional: save to HTML ────────────────────────────────────────────────────
# fig.write_html('bridge_risk_map.html')